In [1]:
import os, sqlite3
import pandas as pd

os.chdir(r"C:\Users\semih\adult-ensemble")
con = sqlite3.connect("adult.db")

train = pd.read_sql_query("SELECT * FROM final_adult WHERE source = 'train'", con)

DROP = ["fnlwgt", "education", "sex", "race", "source"]
X = train.drop(columns=DROP + ["income"])
y = train["income"]

print(X.shape, y.shape)
print("Yüksek gelir oranı:", round(y.mean(), 3))
print(X.isna().sum())

(32537, 10) (32537,)
Yüksek gelir oranı: 0.241
age                  0
workclass         1836
education_num        0
marital_status       0
occupation        1843
relationship         0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     582
dtype: int64


In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

dummy = DummyClassifier(strategy="most_frequent")

sonuc = cross_validate(dummy, X, y, cv=cv, scoring="accuracy")
print("Ortalama:", sonuc["test_score"].mean().round(4))
print("Std      :", sonuc["test_score"].std().round(4))

Ortalama: 0.7591
Std      : 0.0001


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
cat_cols = ["workclass", "marital_status", "occupation", "relationship", "native_country"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

prep = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

Z = prep.fit_transform(X)
print("Dönüşüm sonrası boyut:", Z.shape)

Dönüşüm sonrası boyut: (32537, 84)


In [12]:
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("prep", prep),
    ("model", LogisticRegression(max_iter=1000)),
])

sonuc = cross_validate(
    logreg, X, y, cv=cv,
    scoring=["accuracy", "precision", "recall", "f1"],
)

for metrik in ["accuracy", "precision", "recall", "f1"]:
    s = sonuc[f"test_{metrik}"]
    print(f"{metrik:10s}: {s.mean():.4f} ± {s.std():.4f}")
print("Ortalama eğitim süresi (sn):", sonuc["fit_time"].mean().round(2))

accuracy  : 0.8512 ± 0.0036
precision : 0.7339 ± 0.0091
recall    : 0.6002 ± 0.0113
f1        : 0.6603 ± 0.0091
Ortalama eğitim süresi (sn): 0.19


In [17]:
from sklearn.ensemble import RandomForestClassifier

def degerlendir(isim, model):
    hat = Pipeline([("prep", prep), ("model", model)])
    s = cross_validate(hat, X, y, cv=cv,
                       scoring=["accuracy", "precision", "recall", "f1"])
    print(f"--- {isim} ---")
    for m in ["accuracy", "precision", "recall", "f1"]:
        print(f"{m:10s}: {s[f'test_{m}'].mean():.4f} ± {s[f'test_{m}'].std():.4f}")
    print(f"eğitim süresi (sn): {s['fit_time'].mean():.2f}")
    return s

rf_sonuc = degerlendir("Random Forest", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))

--- Random Forest ---
accuracy  : 0.8480 ± 0.0034
precision : 0.7106 ± 0.0086
recall    : 0.6229 ± 0.0127
f1        : 0.6638 ± 0.0088
eğitim süresi (sn): 2.78


In [18]:
from sklearn.metrics import precision_recall_curve, make_scorer

def prec_at_recall60(y_true, y_score):
    p, r, _ = precision_recall_curve(y_true, y_score)
    return p[r >= 0.60].max()

pr60 = make_scorer(prec_at_recall60, response_method="predict_proba")

def degerlendir(isim, model):
    hat = Pipeline([("prep", prep), ("model", model)])
    s = cross_validate(hat, X, y, cv=cv,
                       scoring={"accuracy": "accuracy", "precision": "precision",
                                "recall": "recall", "f1": "f1", "pr60": pr60})
    print(f"--- {isim} ---")
    for m in ["accuracy", "precision", "recall", "f1", "pr60"]:
        print(f"{m:10s}: {s[f'test_{m}'].mean():.4f} ± {s[f'test_{m}'].std():.4f}")
    print(f"eğitim süresi (sn): {s['fit_time'].mean():.2f}")
    return s

lr_sonuc = degerlendir("Lojistik Regresyon", LogisticRegression(max_iter=1000))
rf_sonuc = degerlendir("Random Forest", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))

--- Lojistik Regresyon ---
accuracy  : 0.8512 ± 0.0036
precision : 0.7339 ± 0.0091
recall    : 0.6002 ± 0.0113
f1        : 0.6603 ± 0.0091
pr60      : 0.7356 ± 0.0144
eğitim süresi (sn): 0.20
--- Random Forest ---
accuracy  : 0.8480 ± 0.0034
precision : 0.7106 ± 0.0086
recall    : 0.6229 ± 0.0127
f1        : 0.6638 ± 0.0088
pr60      : 0.7253 ± 0.0145
eğitim süresi (sn): 2.75


In [19]:
from sklearn.ensemble import GradientBoostingClassifier

gb_sonuc = degerlendir("Gradient Boosting", GradientBoostingClassifier(random_state=42))

--- Gradient Boosting ---
accuracy  : 0.8661 ± 0.0033
precision : 0.7887 ± 0.0092
recall    : 0.6067 ± 0.0114
f1        : 0.6858 ± 0.0088
pr60      : 0.7930 ± 0.0139
eğitim süresi (sn): 2.23


In [20]:
import lightgbm as lgb

lgb_sonuc = degerlendir("LightGBM", lgb.LGBMClassifier(random_state=42, n_jobs=1, verbose=-1))

--- LightGBM ---
accuracy  : 0.8731 ± 0.0031
precision : 0.7818 ± 0.0094
recall    : 0.6570 ± 0.0090
f1        : 0.7139 ± 0.0070
pr60      : 0.8152 ± 0.0109
eğitim süresi (sn): 0.29


In [21]:
sonuclar = {
    "Lojistik Regresyon": lr_sonuc,
    "Random Forest": rf_sonuc,
    "Gradient Boosting": gb_sonuc,
    "LightGBM": lgb_sonuc,
}

satirlar = []
for isim, s in sonuclar.items():
    satirlar.append({
        "model": isim,
        "accuracy": s["test_accuracy"].mean(),
        "precision": s["test_precision"].mean(),
        "recall": s["test_recall"].mean(),
        "pr60": s["test_pr60"].mean(),
        "pr60_std": s["test_pr60"].std(),
        "süre_sn": s["fit_time"].mean(),
    })

tablo = pd.DataFrame(satirlar).set_index("model").round(4)
tablo = tablo.sort_values("pr60", ascending=False)
tablo.to_csv("model_karsilastirma.csv")
display(tablo)

# Eşleştirilmiş kontrol: LightGBM - Gradient Boosting (kat bazında pr60 farkı)
d = lgb_sonuc["test_pr60"] - gb_sonuc["test_pr60"]
print("Ortalama fark:", d.mean().round(4))
print("Farkın std'si:", d.std().round(4))
print("LightGBM'in daha iyi olduğu kat sayısı:", int((d > 0).sum()), "/", len(d))

,accuracy,precision,recall,pr60,pr60_std,süre_sn
model,,,,,,
LightGBM,0.8731,0.7818,0.6570,0.8152,0.0109,0.2941
Gradient Boosting,0.8661,0.7887,0.6067,0.7930,0.0139,2.2339
Lojistik Regresyon,0.8512,0.7339,0.6002,0.7356,0.0144,0.1986
Random Forest,0.8480,0.7106,0.6229,0.7253,0.0145,2.7500


Ortalama fark: 0.0222
Farkın std'si: 0.0086
LightGBM'in daha iyi olduğu kat sayısı: 15 / 15


In [22]:
from scipy.stats import randint, uniform, loguniform
from sklearn.model_selection import RandomizedSearchCV

hat = Pipeline([
    ("prep", prep),
    ("model", lgb.LGBMClassifier(subsample_freq=1, random_state=42, n_jobs=1, verbose=-1)),
])

param_dist = {
    "model__n_estimators": randint(100, 600),
    "model__learning_rate": loguniform(0.02, 0.2),
    "model__num_leaves": randint(8, 64),
    "model__min_child_samples": randint(10, 100),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.5, 0.5),
    "model__reg_lambda": loguniform(0.01, 10),
}

arama = RandomizedSearchCV(
    hat, param_dist, n_iter=30,
    scoring=pr60, cv=cv,
    n_jobs=-1, random_state=42, verbose=1,
)
arama.fit(X, y)

print("En iyi pr60 (CV):", round(arama.best_score_, 4))
print(arama.best_params_)

Fitting 15 folds for each of 30 candidates, totalling 450 fits
En iyi pr60 (CV): 0.8206
{'model__colsample_bytree': np.float64(0.5753587719827147), 'model__learning_rate': np.float64(0.064450868261411), 'model__min_child_samples': 89, 'model__n_estimators': 448, 'model__num_leaves': 10, 'model__reg_lambda': np.float64(0.5924870051994316), 'model__subsample': np.float64(0.871025744736913)}


In [24]:
en_iyi = {k.replace("model__", ""): v for k, v in arama.best_params_.items()}

lgb_ayarli = lgb.LGBMClassifier(subsample_freq=1, random_state=42, n_jobs=1, verbose=-1, **en_iyi)
lgba_sonuc = degerlendir("LightGBM (ayarlı)", lgb_ayarli)

d = lgba_sonuc["test_pr60"] - lgb_sonuc["test_pr60"]
print("Ortalama fark:", d.mean().round(4))
print("Farkın std'si:", d.std().round(4))
print("Ayarlının daha iyi olduğu kat sayısı:", int((d > 0).sum()), "/", len(d))

--- LightGBM (ayarlı) ---
accuracy  : 0.8736 ± 0.0029
precision : 0.7817 ± 0.0086
recall    : 0.6598 ± 0.0077
f1        : 0.7156 ± 0.0064
pr60      : 0.8206 ± 0.0114
eğitim süresi (sn): 0.63
Ortalama fark: 0.0054
Farkın std'si: 0.0053
Ayarlının daha iyi olduğu kat sayısı: 11 / 15


In [25]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (precision_recall_curve, precision_score,
                             recall_score, accuracy_score, f1_score)

final_model = Pipeline([
    ("prep", prep),
    ("model", lgb.LGBMClassifier(random_state=42, n_jobs=1, verbose=-1)),
])

# 1) Eşiği SADECE eğitim verisinden seç (modelin görmediği satırlar için olasılıklar)
oof = cross_val_predict(
    final_model, X, y,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    method="predict_proba",
)[:, 1]

p, r, esikler = precision_recall_curve(y, oof)
idx = np.where(r[:-1] >= 0.60)[0].max()      # recall >= 0,60 olan en yüksek eşik
esik = esikler[idx]
print("Seçilen eşik:", round(esik, 3),
      "| Eğitimde precision:", round(p[idx], 4),
      "| recall:", round(r[idx], 4))

# 2) Test setini yükle, tek seferlik değerlendirme
test = pd.read_sql_query("SELECT * FROM final_adult WHERE source = 'test'", con)
X_test = test.drop(columns=DROP + ["income"])
y_test = test["income"]1

final_model.fit(X, y)
olasilik = final_model.predict_proba(X_test)[:, 1]
tahmin = (olasilik >= esik).astype(int)

print("Test precision:", round(precision_score(y_test, tahmin), 4))
print("Test recall   :", round(recall_score(y_test, tahmin), 4))
print("Test accuracy :", round(accuracy_score(y_test, tahmin), 4))
print("Test f1       :", round(f1_score(y_test, tahmin), 4))

Seçilen eşik: 0.565 | Eğitimde precision: 0.8126 | recall: 0.6001
Test precision: 0.8129
Test recall   : 0.5998
Test accuracy : 0.8729
Test f1       : 0.6903


In [26]:
X_A = train.drop(columns=["fnlwgt", "education", "source", "income"])
cat_cols_A = cat_cols + ["sex", "race"]

prep_A = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols_A),
])
hat_A = Pipeline([
    ("prep", prep_A),
    ("model", lgb.LGBMClassifier(random_state=42, n_jobs=1, verbose=-1)),
])

s_A = cross_validate(hat_A, X_A, y, cv=cv, scoring={"pr60": pr60, "accuracy": "accuracy"})

print("pr60 (sex+race ile)  :", s_A["test_pr60"].mean().round(4), "±", s_A["test_pr60"].std().round(4))
print("pr60 (sex+race yok)  :", lgb_sonuc["test_pr60"].mean().round(4), "±", lgb_sonuc["test_pr60"].std().round(4))

d = s_A["test_pr60"] - lgb_sonuc["test_pr60"]
print("Ortalama fark:", d.mean().round(4), "| farkın std'si:", d.std().round(4))
print("Sütunlar eklenince daha iyi olan kat sayısı:", int((d > 0).sum()), "/", len(d))

pr60 (sex+race ile)  : 0.8151 ± 0.0095
pr60 (sex+race yok)  : 0.8152 ± 0.0109
Ortalama fark: -0.0001 | farkın std'si: 0.0068
Sütunlar eklenince daha iyi olan kat sayısı: 8 / 15
